In [1]:
# %% [markdown]
# # Diagnostic 3: Layer-Wise Activation Distribution Test
# **Objective:** Monitor intermediate layer activations to diagnose saturation and dead neurons.

# %%
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


In [2]:

torch.manual_seed(42)

# 1. Define model with hooks to capture layer outputs
class DiagnosticNet(nn.Module):
    def __init__(self, in_features=32, hidden_dim=64, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.activations = {}

        # Register forward hook on ReLU output
        self.relu.register_forward_hook(self._save_activation("relu_layer"))

    def _save_activation(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# 2. Forward pass over a standardized batch
X = torch.randn(128, 32)
model = DiagnosticNet()
logits = model(X)


In [3]:

act_vals = model.activations["relu_layer"].flatten().cpu().numpy()

# 3. Analyze distribution metrics
dead_neuron_pct = (act_vals == 0.0).mean() * 100
mean_val = act_vals.mean()
std_val = act_vals.std()

print("--- Activation Summary (ReLU) ---")
print(f"Mean Activation: {mean_val:.4f}")
print(f"Std Deviation:   {std_val:.4f}")
print(f"Dead Outputs:    {dead_neuron_pct:.2f}% (zero-valued)")

# 4. Diagnostic inspection
if dead_neuron_pct > 80.0:
    print("WARNING: Severe dying ReLU pathology detected (> 80% dead). Consider LeakyReLU or lower initialization variance.")
elif dead_neuron_pct < 10.0:
    print("PASSED: Low dead neuron ratio. Sufficient gradient flow maintained.")
else:
    print("PASSED: Healthy activation spread.")

--- Activation Summary (ReLU) ---
Mean Activation: 0.2362
Std Deviation:   0.3437
Dead Outputs:    49.65% (zero-valued)
PASSED: Healthy activation spread.


# Diagnostic 3b: Injected Activation Errors

We keep `DiagnosticNet`, the forward hook and the same metrics as above, and inject five realistic errors one at a time. Each run uses the same seed and the same standardized batch unless the error itself is about the input.

| # | Injected error | How |
|---|---|---|
| 1 | **Dead ReLUs from bad bias init** | `fc1.bias` filled with -5 |
| 2 | **Exploding activations (oversized init)** | `fc1.weight ~ N(0, 3^2)` |
| 3 | **Unscaled inputs** (forgot standardization) | `X * 100` |
| 4 | **Vanishing activations (undersized init)** | `fc1.weight ~ N(0, 0.001^2)`, zero bias |
| 5 | **Dying ReLUs from a too-high learning rate** | 30 steps of SGD with `lr=10` |

The original verdict only looks at the fraction of zero outputs. To see what it can and cannot catch, each scenario is scored two ways:

- **Original rule:** exactly the thresholds from the cell above (`> 80%` zeros warns, otherwise passes).
- **Extended rule:** adds a scale check (std outside roughly `[0.05, 3]` for standardized inputs) and a per-unit check for neurons that are zero for every sample.

In [4]:
import numpy as np

def ascii_hist(vals, bins=10, width=36):
    counts, edges = np.histogram(vals, bins=bins)
    top = counts.max()
    for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
        bar = "#" * int(round(width * c / top)) if top > 0 else ""
        print(f"  [{lo:12.4f}, {hi:12.4f}) {bar:<{width}} {100.0 * c / counts.sum():5.1f}%")

def diagnose(label, model, X):
    """Same metrics and original verdict as the baseline cell, plus extended checks."""
    with torch.no_grad():
        model(X)
    act = model.activations["relu_layer"]
    vals = act.flatten().cpu().numpy()

    dead_neuron_pct = (vals == 0.0).mean() * 100          # element-wise zeros (original metric)
    dead_units_pct = (act == 0).all(dim=0).float().mean().item() * 100   # units zero for EVERY sample
    mean_val, std_val = vals.mean(), vals.std()

    print(f"=== {label} ===")
    print(f"Mean Activation: {mean_val:.4f}")
    print(f"Std Deviation:   {std_val:.4g}")
    print(f"Dead Outputs:    {dead_neuron_pct:.2f}% (zero-valued)")
    print(f"Dead Units:      {dead_units_pct:.1f}% (zero for all {act.shape[0]} samples)")

    # Original rule, unchanged
    if dead_neuron_pct > 80.0:
        orig = "WARN"
        print("Original rule: WARNING: Severe dying ReLU pathology detected (> 80% dead).")
    elif dead_neuron_pct < 10.0:
        orig = "PASS"
        print("Original rule: PASSED: Low dead neuron ratio. Sufficient gradient flow maintained.")
    else:
        orig = "PASS"
        print("Original rule: PASSED: Healthy activation spread.")

    # Extended rule
    findings = []
    if dead_neuron_pct > 80.0:
        findings.append("DYING_RELU")
    elif dead_units_pct > 10.0:
        findings.append("DEAD_UNITS")
    if "DYING_RELU" not in findings:
        if std_val > 3.0:
            findings.append("EXPLODING_OR_UNSCALED")
        elif std_val < 0.05:
            findings.append("VANISHING")
    ext = findings if findings else ["HEALTHY"]
    print("Extended rule: " + ", ".join(ext))

    print("Activation histogram:")
    ascii_hist(vals)
    print()
    return dict(label=label, mean=mean_val, std=std_val, dead=dead_neuron_pct,
                dead_units=dead_units_pct, orig=orig, ext=ext)

def build(seed=42):
    """Same construction order as the baseline: seed, standardized batch, then model."""
    torch.manual_seed(seed)
    X = torch.randn(128, 32)
    model = DiagnosticNet()
    return model, X

results = []

In [5]:
# Control: no injected error (reproduces the baseline numbers)
model, X = build()
results.append(diagnose("Control: healthy baseline", model, X))

=== Control: healthy baseline ===
Mean Activation: 0.2362
Std Deviation:   0.3437
Dead Outputs:    49.65% (zero-valued)
Dead Units:      0.0% (zero for all 128 samples)
Original rule: PASSED: Healthy activation spread.
Extended rule: HEALTHY
Activation histogram:
  [      0.0000,       0.2382) ####################################  65.5%
  [      0.2382,       0.4764) #######                               13.5%
  [      0.4764,       0.7146) #####                                  9.9%
  [      0.7146,       0.9528) ###                                    5.8%
  [      0.9528,       1.1909) ##                                     3.3%
  [      1.1909,       1.4291) #                                      1.3%
  [      1.4291,       1.6673)                                        0.5%
  [      1.6673,       1.9055)                                        0.2%
  [      1.9055,       2.1437)                                        0.0%
  [      2.1437,       2.3819)                               

In [6]:
# ERROR 1: dead ReLUs from a bad bias init
model, X = build()
with torch.no_grad():
    model.fc1.bias.fill_(-5.0)          # pushes nearly every pre-activation below zero
results.append(diagnose("Error 1: fc1.bias = -5 (dead ReLUs)", model, X))

=== Error 1: fc1.bias = -5 (dead ReLUs) ===
Mean Activation: 0.0000
Std Deviation:   0
Dead Outputs:    100.00% (zero-valued)
Dead Units:      100.0% (zero for all 128 samples)
Original rule: WARNING: Severe dying ReLU pathology detected (> 80% dead).
Extended rule: DYING_RELU
Activation histogram:
  [     -0.5000,      -0.4000)                                        0.0%
  [     -0.4000,      -0.3000)                                        0.0%
  [     -0.3000,      -0.2000)                                        0.0%
  [     -0.2000,      -0.1000)                                        0.0%
  [     -0.1000,       0.0000)                                        0.0%
  [      0.0000,       0.1000) #################################### 100.0%
  [      0.1000,       0.2000)                                        0.0%
  [      0.2000,       0.3000)                                        0.0%
  [      0.3000,       0.4000)                                        0.0%
  [      0.4000,       0.

In [7]:
# ERROR 2: oversized weight init -> exploding activations
model, X = build()
nn.init.normal_(model.fc1.weight, std=3.0)
results.append(diagnose("Error 2: fc1.weight std = 3.0 (exploding)", model, X))

=== Error 2: fc1.weight std = 3.0 (exploding) ===
Mean Activation: 6.6004
Std Deviation:   9.837
Dead Outputs:    50.56% (zero-valued)
Dead Units:      0.0% (zero for all 128 samples)
Original rule: PASSED: Healthy activation spread.
Extended rule: EXPLODING_OR_UNSCALED
Activation histogram:
  [      0.0000,       6.9559) ####################################  66.9%
  [      6.9559,      13.9118) #######                               12.9%
  [     13.9118,      20.8677) #####                                  9.4%
  [     20.8677,      27.8236) ###                                    5.7%
  [     27.8236,      34.7795) ##                                     3.1%
  [     34.7795,      41.7354) #                                      1.2%
  [     41.7354,      48.6913)                                        0.4%
  [     48.6913,      55.6472)                                        0.2%
  [     55.6472,      62.6031)                                        0.1%
  [     62.6031,      69.5590)  

In [8]:
# ERROR 3: forgot to standardize the inputs
model, X = build()
X = X * 100.0                           # features on a much larger scale than the init assumes
results.append(diagnose("Error 3: unscaled inputs (X * 100)", model, X))

=== Error 3: unscaled inputs (X * 100) ===
Mean Activation: 23.0944
Std Deviation:   33.7
Dead Outputs:    49.93% (zero-valued)
Dead Units:      0.0% (zero for all 128 samples)
Original rule: PASSED: Healthy activation spread.
Extended rule: EXPLODING_OR_UNSCALED
Activation histogram:
  [      0.0000,      23.0817) ####################################  65.5%
  [     23.0817,      46.1635) #######                               13.2%
  [     46.1635,      69.2452) ######                                10.0%
  [     69.2452,      92.3270) ###                                    5.7%
  [     92.3270,     115.4087) ##                                     3.2%
  [    115.4087,     138.4904) #                                      1.4%
  [    138.4904,     161.5722)                                        0.5%
  [    161.5722,     184.6539)                                        0.3%
  [    184.6539,     207.7357)                                        0.0%
  [    207.7357,     230.8174)         

In [9]:
# ERROR 4: undersized weight init -> vanishing activations
model, X = build()
nn.init.normal_(model.fc1.weight, std=1e-3)
nn.init.zeros_(model.fc1.bias)
results.append(diagnose("Error 4: fc1.weight std = 0.001 (vanishing)", model, X))

=== Error 4: fc1.weight std = 0.001 (vanishing) ===
Mean Activation: 0.0022
Std Deviation:   0.003279
Dead Outputs:    50.65% (zero-valued)
Dead Units:      0.0% (zero for all 128 samples)
Original rule: PASSED: Healthy activation spread.
Extended rule: VANISHING
Activation histogram:
  [      0.0000,       0.0023) ####################################  66.8%
  [      0.0023,       0.0046) #######                               13.1%
  [      0.0046,       0.0070) #####                                  9.3%
  [      0.0070,       0.0093) ###                                    5.7%
  [      0.0093,       0.0116) ##                                     3.2%
  [      0.0116,       0.0139) #                                      1.2%
  [      0.0139,       0.0162)                                        0.4%
  [      0.0162,       0.0185)                                        0.2%
  [      0.0185,       0.0209)                                        0.1%
  [      0.0209,       0.0232)         

In [10]:
# ERROR 5: learning rate far too high -> ReLUs die during training
model, X = build()
y = torch.randint(0, 5, (128,))
opt = torch.optim.SGD(model.parameters(), lr=10.0)    # should be ~0.01-0.1
crit = nn.CrossEntropyLoss()
for step in range(30):
    opt.zero_grad()
    crit(model(X), y).backward()
    opt.step()
results.append(diagnose("Error 5: SGD lr = 10 for 30 steps (dying ReLUs)", model, X))

=== Error 5: SGD lr = 10 for 30 steps (dying ReLUs) ===
Mean Activation: 0.1241
Std Deviation:   0.6419
Dead Outputs:    93.59% (zero-valued)
Dead Units:      18.8% (zero for all 128 samples)
Original rule: WARNING: Severe dying ReLU pathology detected (> 80% dead).
Extended rule: DYING_RELU
Activation histogram:
  [      0.0000,       0.9227) ####################################  95.9%
  [      0.9227,       1.8453) #                                      1.5%
  [      1.8453,       2.7680)                                        0.9%
  [      2.7680,       3.6907)                                        0.6%
  [      3.6907,       4.6133)                                        0.5%
  [      4.6133,       5.5360)                                        0.3%
  [      5.5360,       6.4586)                                        0.2%
  [      6.4586,       7.3813)                                        0.1%
  [      7.3813,       8.3040)                                        0.0%
  [      8

In [11]:
# Summary: which errors does each rule catch?
print(f"{'Scenario':<50} {'mean':>10} {'std':>10} {'dead%':>7} {'deadU%':>7}  {'Original':<8} Extended")
print("-" * 115)
for r in results:
    print(f"{r['label']:<50} {r['mean']:>10.4g} {r['std']:>10.4g} {r['dead']:>7.1f} {r['dead_units']:>7.1f}  {r['orig']:<8} {', '.join(r['ext'])}")

errors = results[1:]
missed = [r["label"] for r in errors if r["orig"] == "PASS"]
caught_ext = [r["label"] for r in errors if r["ext"] != ["HEALTHY"]]
print()
print(f"Original rule caught {len(errors) - len(missed)} of {len(errors)} injected errors.")
print(f"Extended rule caught {len(caught_ext)} of {len(errors)} injected errors.")
print("Missed by the original rule:")
for m in missed:
    print("  -", m)

Scenario                                                 mean        std   dead%  deadU%  Original Extended
-------------------------------------------------------------------------------------------------------------------
Control: healthy baseline                              0.2362     0.3437    49.6     0.0  PASS     HEALTHY
Error 1: fc1.bias = -5 (dead ReLUs)                         0          0   100.0   100.0  WARN     DYING_RELU
Error 2: fc1.weight std = 3.0 (exploding)                 6.6      9.837    50.6     0.0  PASS     EXPLODING_OR_UNSCALED
Error 3: unscaled inputs (X * 100)                      23.09       33.7    49.9     0.0  PASS     EXPLODING_OR_UNSCALED
Error 4: fc1.weight std = 0.001 (vanishing)            0.0022   0.003279    50.6     0.0  PASS     VANISHING
Error 5: SGD lr = 10 for 30 steps (dying ReLUs)        0.1241     0.6419    93.6    18.8  WARN     DYING_RELU

Original rule caught 2 of 5 injected errors.
Extended rule caught 5 of 5 injected errors.
Missed 

## Takeaways

- **Element-wise zeros of about 50% are normal for a ReLU at init.** The baseline "49.65% dead" is healthy because the zeros are spread across units and samples. The signal that matters is **units that are zero for every sample** (the `Dead Units` line) or a zero fraction that moves far above 50%.
- **The dead-percentage rule only catches dying ReLUs.** Errors 1 and 5 trip it. Errors 2, 3 and 4 all leave about half the outputs at zero, so the original verdict still says PASSED even though the activation std is about 30x too large (error 2), about 100x too large (error 3), or about 100x too small (error 4) compared with the control's 0.34.
- **Scale errors need the mean and std.** Activation std is the cheapest way to see exploding, unscaled or vanishing activations. Compare it with the input scale: standardized inputs through a sane init should give a std of order 0.1 to 1.
- **Read the histogram's bin edges, not just the bars.** In errors 2, 3 and 4 the bar pattern is identical to the control, with the same spike at zero and the same tail. Only the axis range changes (about 0 to 60, 0 to 230, or 0 to 0.02 versus 0 to 2). A shape-only plot would miss them, which is why the numeric std check matters. Errors 1 and 5 do change the shape, with the mass collapsing onto zero.
- **Error 5 is a training-time failure.** The same model with a healthy init gets 93.6% zeros and about a fifth of its units permanently dead after a few oversized steps. Run this check after a few training steps as well as at init.
- **Fixes:** use a He/Kaiming init for ReLU nets, standardize inputs, keep biases near zero, lower the learning rate, and consider LeakyReLU or normalization layers if units keep dying.